In [1]:
from pathlib import Path 
import polars as pl 
import matplotlib.pyplot as plt


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"

recs = pl.scan_parquet(DATA / "recommendations.parquet")   
games = pl.read_parquet(DATA / "games.parquet")
recs.collect_schema()

Schema([('app_id', Int32),
        ('helpful', Int64),
        ('funny', Int64),
        ('date', Date),
        ('is_recommended', Boolean),
        ('hours', Float32),
        ('user_id', Int32),
        ('review_id', Int64)])

In [2]:
overview = recs.select(
    pl.len().alias("reviews"),
    pl.col("user_id").n_unique().alias("users"),
    pl.col("app_id").n_unique().alias("games"),
    pl.col("date").min().alias("first_review"),
    pl.col("date").max().alias("last_review"),
    pl.col("is_recommended").mean().alias("share_recommended"),
    pl.col("hours").mean().alias("avg_hours"),
    pl.col("hours").median().alias("median_hours"),
).collect()

overview

reviews,users,games,first_review,last_review,share_recommended,avg_hours,median_hours
u32,u32,u32,date,date,f64,f32,f32
41154794,13781059,37610,2010-10-15,2022-12-31,0.857844,100.602242,27.299999


In [3]:
recs = pl.scan_parquet(DATA / "recommendations.parquet")
games = pl.read_parquet(DATA / "games.parquet")
users = pl.read_parquet(DATA / "users.parquet")
meta = pl.read_parquet(DATA / "games_metadata.parquet")

In [4]:
recs.head(10).collect()


app_id,helpful,funny,date,is_recommended,hours,user_id,review_id
i32,i64,i64,date,bool,f32,i32,i64
975370,0,0,2022-12-12,true,36.299999,51580,0
304390,4,0,2017-02-17,false,11.5,2586,1
1085660,2,0,2019-11-17,true,336.5,253880,2
703080,0,0,2022-09-23,true,27.4,259432,3
526870,0,0,2021-01-10,true,7.9,23869,4
306130,0,0,2021-10-10,true,8.6,45425,5
238960,0,0,2017-11-25,true,538.799988,88282,6
730,0,0,2021-11-30,false,157.5,63209,7
255710,0,0,2021-05-21,true,18.700001,354512,8


In [5]:
games.filter(pl.col("title").str.contains("DLC"))

app_id,title,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,price_original,discount,steam_deck
i64,str,date,bool,bool,bool,str,i64,i64,f64,f64,f64,bool
462280,"""Offworld Trading Company - Sou…",2016-04-28,true,true,false,"""Positive""",80,10,9.99,9.99,0.0,true
439500,"""LEGO® MARVEL's Avengers DLC - …",2016-03-29,true,false,false,"""Positive""",93,16,2.99,2.99,0.0,true
584096,"""Little Nightmares The Hideaway…",2017-11-09,true,false,false,"""Mostly Positive""",76,309,3.99,3.99,0.0,true
292050,"""Valkyrie Hat (or 'Buy Us Coffe…",2015-07-13,true,false,false,"""Positive""",100,43,4.99,4.99,0.0,true
1906530,"""A Plague Tale: Requiem - Prote…",2022-10-17,true,false,false,"""Positive""",94,35,2.99,2.99,0.0,true
…,…,…,…,…,…,…,…,…,…,…,…,…
715890,"""Sanic T-Shirts DLC""",2017-11-27,true,false,false,"""Very Positive""",83,106,0.0,0.0,0.0,true
1718780,"""Circuit Superstars DLC: Top Ge…",2021-10-12,true,false,false,"""Positive""",100,17,0.0,0.0,0.0,true
2464230,"""Car Mechanic Simulator 2021 - …",2023-06-29,true,false,false,"""Mostly Positive""",73,15,15.99,15.99,0.0,true


In [6]:
recs.count().collect()

app_id,helpful,funny,date,is_recommended,hours,user_id,review_id
u32,u32,u32,u32,u32,u32,u32,u32
41154794,41154794,41154794,41154794,41154794,41154794,41154794,41154794


## Data quality checks

### Row count: CSV vs parquet
Confirms the parquet conversion didn't drop or duplicate any rows.

In [7]:
csv_rows = pl.scan_csv(ROOT / "data" / "raw" / "recommendations.csv").select(pl.len()).collect().item()
parquet_rows = recs.select(pl.len()).collect().item()

assert csv_rows == parquet_rows, f"Mismatch: CSV {csv_rows:,} vs parquet {parquet_rows:,}"
print(f"Row counts match: {parquet_rows:,}")

Row counts match: 41,154,794


### Missing values per column

In [8]:
recs.null_count().collect()

app_id,helpful,funny,date,is_recommended,hours,user_id,review_id
u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0


### Duplicate (user_id, app_id) pairs
A user should review each game at most once. Any pair appearing more than once is a duplicate.

In [9]:
dupes = (
    recs.group_by(["user_id", "app_id"])
    .len()
    .filter(pl.col("len") > 1)
    .collect(engine="streaming")
)

print(f"Duplicate pairs: {dupes.height:,}")
print(f"Extra rows they add: {(dupes['len'] - 1).sum():,}")
dupes.head()

Duplicate pairs: 21
Extra rows they add: 21


user_id,app_id,len
i32,i32,u32
3054178,1454400,2
3148709,1063730,2
5485206,220,2
5235993,266840,2
3070422,1118200,2


### Hours played
Range of `hours`, and how many reviews were written with 0 hours (split by recommended / not recommended).

In [10]:
recs.select(
    pl.col("hours").min().alias("min_hours"),
    pl.col("hours").max().alias("max_hours"),
    (pl.col("hours") == 0).sum().alias("zero_hours"),
    ((pl.col("hours") == 0) & pl.col("is_recommended")).sum().alias("zero_hours_recommended"),
    (pl.col("hours") == 0).mean().alias("share_zero_hours"),
).collect()

min_hours,max_hours,zero_hours,zero_hours_recommended,share_zero_hours
f32,f32,u32,u32,f64
0.0,1000.0,176329,105703,0.004285
